In  a docker terminal run
jupyter notebook --ip=0.0.0.0 --port=8888 --no-browser --allow-root

then open jupyter in the printed link (such as http://127.0.0.1:8888/tree?token=4d9d578397aede50fc5bd2d92794b562a6bbcbc73b2dfe51)

CODE HERE, REFRESH AND EXECUTE IN THE BROWSER

In [1]:
import os
import sys
import django

os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

sys.path.append('/app')

os.environ['DJANGO_SETTINGS_MODULE'] = 'app.settings'
os.environ['PYTHONPATH'] = '/app'

django.setup()


In [2]:
from django.conf import settings
import requests
from requests.auth import HTTPBasicAuth
import json
import rasterio
import numpy as np

In [7]:
filename= 'Entwaesserungswahrscheinlichkeit_9Parameter'
with rasterio.open(filename + '.tif') as f:
    vals = f.read(1)
    width = f.width
    height = f.height
    transform = f.transform
    bounds=f.bounds
    nodata=f.nodata
    dst_profile=f.profile.copy()
    crs=f.crs
    print(dir(f))
    

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__enter__', '__eq__', '__exit__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pyx_vtable__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_block_shapes', '_closed', '_count', '_crs', '_crs_wkt', '_descriptions', '_dtypes', '_env', '_gcps', '_get_crs', '_get_rpcs', '_handle_crswkt', '_has_band', '_has_gcps_or_rpcs', '_mask_flags', '_nodatavals', '_offsets', '_read', '_rpcs', '_scales', '_set_all_descriptions', '_set_all_offsets', '_set_all_scales', '_set_all_units', '_set_attrs_from_dataset_handle', '_set_crs', '_set_gcps', '_set_nodatavals', '_set_rpcs', '_transform', '_units', 'block_shapes', 'block_size', 'block_window', 'block_windows', 'bounds', 'checksum', 'close', 'closed', 'colorinterp', 'colormap', 'compression', 'c

In [27]:
dst_profile

{'driver': 'GTiff', 'dtype': 'float32', 'nodata': -3.4028234663852886e+38, 'width': 6425, 'height': 9474, 'count': 1, 'crs': CRS.from_wkt('PROJCS["ETRS89 / UTM zone 33N",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",15],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","25833"]]'), 'transform': Affine(10.0, 0.0, 401961.0594,
       0.0, -10.0, 5844979.2294), 'blockxsize': 6425, 'blockysize': 1, 'tiled': False, 'interleave': 'band'}

In [15]:
dst_profile['nodata'] = dnodata


In [16]:
vals[vals==nodata]=dnodata
# vals[vals==vals.min()]=np.nan

In [24]:
vals = np.float32(vals)

In [40]:
vals[vals==nodata]=np.nan
vals[vals==vals.min()]=np.nan

In [41]:
dst_profile['nodata'] = np.nan

In [28]:
new_filename = filename + '_v2.tif'
with rasterio.open(new_filename, 'w', **dst_profile) as f:
    f.write(vals.astype(np.float32),1)
    
    


In [78]:
# RGBA to singleband

filename= 'dgm200'
with rasterio.open(filename + '.tif') as f:
    vals = f.read()
    width = f.width
    height = f.height
    transform = f.transform
    bounds=f.bounds
    nodata=f.nodata
    dst_profile=f.profile.copy()
    crs=f.crs
    print(dir(f))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__enter__', '__eq__', '__exit__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pyx_vtable__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_block_shapes', '_closed', '_count', '_crs', '_crs_wkt', '_descriptions', '_dtypes', '_env', '_gcps', '_get_crs', '_get_rpcs', '_handle_crswkt', '_has_band', '_has_gcps_or_rpcs', '_mask_flags', '_nodatavals', '_offsets', '_read', '_rpcs', '_scales', '_set_all_descriptions', '_set_all_offsets', '_set_all_scales', '_set_all_units', '_set_attrs_from_dataset_handle', '_set_crs', '_set_gcps', '_set_nodatavals', '_set_rpcs', '_transform', '_units', 'block_shapes', 'block_size', 'block_window', 'block_windows', 'bounds', 'checksum', 'close', 'closed', 'colorinterp', 'colormap', 'compression', 'c

In [79]:
dem = np.full_like(vals[1], np.nan, dtype=np.float32)
scale_factor = 3000 / 255
dem[vals[3] != 0] = vals[1][vals[3] != 0] * scale_factor

In [80]:
dst_profile['nodata'] = np.nan

In [81]:
dst_profile['count']=1
dst_profile['dtype']='float32'

In [82]:
with rasterio.open('dgm200_single_v1.tif', 'w', **dst_profile) as f:
    f.write(dem.astype(np.float32),1)

In [83]:
filename= 'dgm200_single_v1'
with rasterio.open(filename + '.tif') as f:
    vals = f.read()
    width = f.width
    height = f.height
    transform = f.transform
    bounds=f.bounds
    nodata=f.nodata
    dst_profile=f.profile.copy()
    crs=f.crs
    print(dir(f))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__enter__', '__eq__', '__exit__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pyx_vtable__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_block_shapes', '_closed', '_count', '_crs', '_crs_wkt', '_descriptions', '_dtypes', '_env', '_gcps', '_get_crs', '_get_rpcs', '_handle_crswkt', '_has_band', '_has_gcps_or_rpcs', '_mask_flags', '_nodatavals', '_offsets', '_read', '_rpcs', '_scales', '_set_all_descriptions', '_set_all_offsets', '_set_all_scales', '_set_all_units', '_set_attrs_from_dataset_handle', '_set_crs', '_set_gcps', '_set_nodatavals', '_set_rpcs', '_transform', '_units', 'block_shapes', 'block_size', 'block_window', 'block_windows', 'bounds', 'checksum', 'close', 'closed', 'colorinterp', 'colormap', 'compression', 'c

In [85]:
dst_profile

{'driver': 'GTiff', 'dtype': 'float32', 'nodata': nan, 'width': 3211, 'height': 4331, 'count': 1, 'crs': CRS.from_wkt('PROJCS["ETRS89 / UTM zone 32N",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",9],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","25832"]]'), 'transform': Affine(200.0, 0.0, 279900.0,
       0.0, -200.0, 6101900.0), 'blockxsize': 3211, 'blockysize': 1, 'tiled': False, 'interleave': 'band'}

In [6]:
filename= 'distance_to_extraction_wells_v1'
with rasterio.open(filename + '.tif') as f:
    dvals = f.read()
    dwidth = f.width
    dheight = f.height
    dtransform = f.transform
    dbounds=f.bounds
    dnodata=f.nodata
    ddst_profile=f.profile.copy()
    dcrs=f.crs
    print(dir(f))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__enter__', '__eq__', '__exit__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pyx_vtable__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_block_shapes', '_closed', '_count', '_crs', '_crs_wkt', '_descriptions', '_dtypes', '_env', '_gcps', '_get_crs', '_get_rpcs', '_handle_crswkt', '_has_band', '_has_gcps_or_rpcs', '_mask_flags', '_nodatavals', '_offsets', '_read', '_rpcs', '_scales', '_set_all_descriptions', '_set_all_offsets', '_set_all_scales', '_set_all_units', '_set_attrs_from_dataset_handle', '_set_crs', '_set_gcps', '_set_nodatavals', '_set_rpcs', '_transform', '_units', 'block_shapes', 'block_size', 'block_window', 'block_windows', 'bounds', 'checksum', 'close', 'closed', 'colorinterp', 'colormap', 'compression', 'c